In [1]:
import json
import re
import codecs
import urllib.request
import requests 
from bs4 import BeautifulSoup # for HTML parsing
import time # for sleep
from tqdm import tqdm # for progress tracker
import urllib3 # to disable SSL warnings 

# Disable the SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# Opens and reads the JSON file gained from JT's data vis scraping tool 
with open('articles-and-dvcs.json', 'r') as f:    
    results = json.load(f)
    

# Only process the latest 10 articles
results = sorted(results, key=lambda x: x['release_date'], reverse=True)[:10]



In [3]:
# Load existing data.json to merge into, rather than starting fresh
try:
    with open('data.json', 'r') as f:
        existing_data = json.load(f)
except FileNotFoundError:
    existing_data = []

# Build a lookup of existing articles by title for upsert logic
pages_lookup = {article['title']: article for article in existing_data}

# pages will hold only newly processed articles from this run
pages = []

In [4]:
for article in tqdm(results, desc="Processing articles"):

    #get the article theme from the url
    split = article['doc_uri'].split('/')
    article['theme'] = split[3]

    if article['vis_urls']:
        article['vis'] = []
        for i, vis in enumerate(tqdm(article['vis_urls'], desc="Fetching visualizations", leave=False)):
            if 'datadownload.xlsx' in vis:
                continue
            url = "https://www.ons.gov.uk" + vis
            html = requests.get(url, verify=False).text

            # Only sleep between requests, not after the last one
            # if i < len(article['vis_urls']) - 1: 
            #     time.sleep(0.5)
            
            soup = BeautifulSoup(html, "lxml")
    
# Most common: <meta name="template" content="...">
            meta = soup.find("meta", attrs={"name": "template"})
            template = meta.get("content") if meta else None
            article["vis"].append({"url": vis, "chart_type": template})

# Fallback: any meta where name/property contains "template"
            if template is None:
                meta = soup.find("meta", attrs={"name": lambda v: v and "template" in v.lower()}) \
                    or soup.find("meta", attrs={"property": lambda v: v and "template" in v.lower()})
                template = meta.get("content") if meta else None
            # print(template)

            # if this release is not currently in the list pages add it
        if not pages:
                pages.append(article)

            # if there is a previous version of this article replace it with the newer version if it has data vis in it
        elif article['vis_urls']:
            article_titles = list(map(lambda article: article['title'], pages))
            if article['title'] in article_titles:
                for index, item in enumerate(pages):
                    if item['title'] == article['title']:
                        item = article
                        break
                    else:
                        index = -1
            else:
                pages.append(article)

Processing articles: 100%|██████████| 10/10 [00:08<00:00,  1.18it/s]


In [5]:
# Upsert newly processed articles into the existing data lookup
for article in pages:
    pages_lookup[article['title']] = article

# Convert lookup back to a list and sort by release date, newest first
merged = list(pages_lookup.values())
merged.sort(key=lambda x: x['release_date'], reverse=True)

json_str = json.dumps(merged, indent=6)
with open("data.json", "w") as f:
    f.write(json_str)